# 03 SOKE20  Stage-1.5 Continuous Refiner

 continuous residual refiner for the SOKE20 multi-dataset tokenizer. Stage 1 is frozen from the trained `t08_s32_cd1024_cn1024_b32_ga1_soke20` tokenizer; this notebook caches full-clip stage1 reconstructions, trains a temporal refiner in 277-D canonical space, and can run validation motion metrics.

In [ ]:
from pathlib import Path
import os, shlex, subprocess, sys

ROOT = Path('/home/cem/tez/exp') if Path('/home/cem/tez/exp').exists() else Path.cwd()
NB_ROOT = ROOT / 'new_data' / '03_soke20_nb51_stage2_refiner'
STAGE1_ROOT = ROOT / 'new_data' / '02_soke20fps_nb51_tokenizer'
PREPROCESS_ROOT = STAGE1_ROOT / 'outputs' / 'preprocess_soke20'
STAGE1_RUN_ROOT = STAGE1_ROOT / 'outputs' / 'runs' / 't08_s32_cd1024_cn1024_b32_ga1_soke20'
RUN_ROOT = NB_ROOT / 'outputs' / 'runs' / 'refiner_on_t08_s32_cd1024_cn1024_b32_ga1_soke20'
SCRIPT = NB_ROOT / 'scripts' / 'train_soke20_nb51_refiner.py'

# Edit these first. Use 0 for full train/val manifests.
EPOCHS = int(os.environ.get('SOKE20_REFINER_EPOCHS', '200'))
BATCH_SIZE = int(os.environ.get('SOKE20_REFINER_BATCH_SIZE', '32'))
NUM_WORKERS = int(os.environ.get('SOKE20_REFINER_NUM_WORKERS', '2'))
WINDOWS_PER_CLIP = int(os.environ.get('SOKE20_REFINER_WINDOWS_PER_CLIP', '1'))
MAX_TRAIN_CLIPS = int(os.environ.get('SOKE20_REFINER_MAX_TRAIN_CLIPS', '0'))
MAX_VAL_CLIPS = int(os.environ.get('SOKE20_REFINER_MAX_VAL_CLIPS', '0'))
REBUILD_RECON_CACHE = os.environ.get('SOKE20_REFINER_REBUILD_RECON_CACHE', '0') == '1'
REBUILD_STATS = os.environ.get('SOKE20_REFINER_REBUILD_STATS', '0') == '1'
RESUME = int(os.environ.get('SOKE20_REFINER_RESUME', '1'))
RUN_MOTION_METRICS = int(os.environ.get('SOKE20_REFINER_RUN_MOTION_METRICS', '0'))
METRIC_MAX_CLIPS = int(os.environ.get('SOKE20_REFINER_METRIC_MAX_CLIPS', '0'))

# Previous best refiner defaults from notebooks 39/40.
D_MODEL = int(os.environ.get('SOKE20_REFINER_D_MODEL', '256'))
NUM_HEADS = int(os.environ.get('SOKE20_REFINER_NUM_HEADS', '8'))
NUM_KV_HEADS = int(os.environ.get('SOKE20_REFINER_NUM_KV_HEADS', '4'))
SPATIAL_IN_BLOCKS = int(os.environ.get('SOKE20_REFINER_SPATIAL_IN_BLOCKS', '2'))
TEMPORAL_BLOCKS = int(os.environ.get('SOKE20_REFINER_TEMPORAL_BLOCKS', '2'))
SPATIAL_OUT_BLOCKS = int(os.environ.get('SOKE20_REFINER_SPATIAL_OUT_BLOCKS', '2'))
LR = float(os.environ.get('SOKE20_REFINER_LR', '2e-4'))
EARLY_STOP_PATIENCE = int(os.environ.get('SOKE20_REFINER_EARLY_STOP_PATIENCE', '50'))

print({
    'ROOT': str(ROOT),
    'PREPROCESS_ROOT': str(PREPROCESS_ROOT),
    'STAGE1_RUN_ROOT': str(STAGE1_RUN_ROOT),
    'RUN_ROOT': str(RUN_ROOT),
    'EPOCHS': EPOCHS,
    'BATCH_SIZE': BATCH_SIZE,
    'MAX_TRAIN_CLIPS': MAX_TRAIN_CLIPS,
    'MAX_VAL_CLIPS': MAX_VAL_CLIPS,
    'RUN_MOTION_METRICS': RUN_MOTION_METRICS,
})

In [ ]:
cmd = [
    sys.executable, str(SCRIPT),
    '--preprocess-root', str(PREPROCESS_ROOT),
    '--stage1-run-root', str(STAGE1_RUN_ROOT),
    '--run-root', str(RUN_ROOT),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--windows-per-clip', str(WINDOWS_PER_CLIP),
    '--max-train-clips', str(MAX_TRAIN_CLIPS),
    '--max-val-clips', str(MAX_VAL_CLIPS),
    '--d-model', str(D_MODEL),
    '--num-heads', str(NUM_HEADS),
    '--num-kv-heads', str(NUM_KV_HEADS),
    '--spatial-in-blocks', str(SPATIAL_IN_BLOCKS),
    '--temporal-blocks', str(TEMPORAL_BLOCKS),
    '--spatial-out-blocks', str(SPATIAL_OUT_BLOCKS),
    '--lr', str(LR),
    '--early-stop-patience', str(EARLY_STOP_PATIENCE),
    '--resume', str(RESUME),
    '--run-motion-metrics', str(RUN_MOTION_METRICS),
    '--metric-max-clips', str(METRIC_MAX_CLIPS),
]
if REBUILD_RECON_CACHE:
    cmd.append('--rebuild-recon-cache')
if REBUILD_STATS:
    cmd.append('--rebuild-stats')
print(' '.join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)

In [ ]:
import pandas as pd
from IPython.display import Image, display

hist = RUN_ROOT / 'history.csv'
summary = RUN_ROOT / 'results_summary.csv'
plot = RUN_ROOT / 'plots' / 'refiner_training_curves.png'
if hist.exists():
    display(pd.read_csv(hist).tail())
if summary.exists():
    display(pd.read_csv(summary))
if plot.exists():
    display(Image(filename=str(plot)))

In [ ]:
# Optional: run validation motion metrics after a best refiner checkpoint exists.
if (RUN_ROOT / 'best.pt').exists() and RUN_MOTION_METRICS == 0:
    metric_cmd = [
        sys.executable, str(SCRIPT),
        '--preprocess-root', str(PREPROCESS_ROOT),
        '--stage1-run-root', str(STAGE1_RUN_ROOT),
        '--run-root', str(RUN_ROOT),
        '--skip-recon-cache', '--skip-training',
        '--run-motion-metrics', '1',
        '--metric-max-clips', str(METRIC_MAX_CLIPS),
    ]
    print('To run metrics:')
    print(' '.join(shlex.quote(x) for x in metric_cmd))
else:
    print('No best checkpoint yet, or metrics already requested in the training command.')

In [ ]:
# Colab billing guard. Set SOKE20_REFINER_KEEP_RUNTIME=1 to inspect the live runtime.
if os.environ.get('SOKE20_REFINER_KEEP_RUNTIME', '0') != '1':
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as exc:
        print('Runtime close skipped:', exc)